# RNO-G Simulation Validation Tools

A set of validation plotting tools designed for quick and accessible inspection of RNO-G simulation sets.

## Key Features & Information
- **Energy Distribution:** Spectrum analysis and primary energy tracking
- **Flavor Information:** Neutrino flavor breakdown
- **Effective Volume ($V_{\text{eff}}$):** Calculation and distribution metrics
- **Kinematics:** Launching angle and interaction vertex positioning
- **Inelasticity ($y$):** Inelasticity distribution per event

> **Note:** This module processes **1 shower per event** (selecting the primary/first shower to avoid double-counting secondary interactions).

In [ ]:

# imports
from matplotlib import pyplot as plt
import numpy as np
import h5py
import math
import datetime

# NuRadioMC imports
from NuRadioReco.utilities import units
from NuRadioMC.utilities.Veff import (
    get_Veff_water_equivalent,
    remove_duplicate_triggers,
)
import NuRadioReco.detector.RNO_G.rnog_detector
import NuRadioReco.utilities.units as units
import NuRadioReco.modules.RNO_G.dataProviderRNOG

# This notebook is configured for merged continuous-spectrum validation.

In [ ]:
# Data reading block, use combined hdf5 file below:
h5_path = ""

with h5py.File(h5_path, "r") as fin:
    energies = np.asarray(fin["energies"], dtype=float)        # eV
    sh_energies = np.asarray(fin["shower_energies"], dtype=float)
    weights = np.asarray(fin["weights"], dtype=float)
    flavors = np.asarray(fin["flavors"], dtype=int)
    triggered = np.asarray(fin["triggered"], dtype=bool)
    event_group_ids = np.asarray(fin["event_group_ids"])
    inelasticity=np.asarray(fin["inelasticity"],dtype=float)

    xx=np.asarray(fin["xx"], dtype=float)
    yy=np.asarray(fin["yy"], dtype=float)
    zz=np.asarray(fin["zz"], dtype=float)
    zeniths=np.asarray(fin["zeniths"], dtype=float)
    azimuths=np.asarray(fin["azimuths"], dtype=float)
    
# "Passed" events = triggered events (deduplicated per event group)
# without remove_duplicate_triggers, every secondary particle shower will be count as a seperate signals
passed_mask = remove_duplicate_triggers(triggered, event_group_ids)

E_passed = energies[passed_mask]
sh_E_passed = sh_energies[passed_mask]
F_passed = np.abs(flavors[passed_mask]).astype(int)
w_passed = weights[passed_mask]
inelasticity_passed=inelasticity[passed_mask]

print(f"Total events: {len(energies)}")
print(f"Passed events (deduplicated): {len(E_passed)}")
print(f"Total passed weight: {w_passed.sum():.3f}")


In [ ]:
# Log-spaced bins over passed-energy range
n_bins = 30
bins = np.logspace(np.log10(E_passed.min()), np.log10(E_passed.max()), n_bins + 1)

# Weighted histogram
hist_w, edges = np.histogram(E_passed, bins=bins, weights=w_passed)
centers = np.sqrt(edges[:-1] * edges[1:])  # geometric bin center

plt.figure(figsize=(8, 5))
plt.step(centers, hist_w, where="mid", linewidth=2, label="Passed weighted events")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Neutrino energy [eV]")
plt.ylabel("Weighted counts per bin")
plt.title("Energy distribution of passed weighted events")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
n_bins = 12
energy_bins = np.logspace(np.log10(E_passed.min()), np.log10(E_passed.max()), n_bins + 1)
energy_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

flavor_ids = (12, 13, 14, 15, 16)
flavor_labels = {
    12: "12: electron neutrino",
    13: "13: muon",
    14: "14: muon neutrino",
    15: "15: tau",
    16: "16: tau neutrino",
}
flavor_markers = {12: "o", 13: "v", 14: "s", 15: "D", 16: "^"}

weighted_hists = {}
for fid in flavor_ids:
    mask = F_passed == fid
    weighted_hists[fid], _ = np.histogram(E_passed[mask], bins=energy_bins, weights=w_passed[mask])

total_weighted = np.sum(np.vstack([weighted_hists[fid] for fid in flavor_ids]), axis=0)
safe_total = np.where(total_weighted > 0, total_weighted, 1.0)

plt.figure(figsize=(8, 5))
for fid in flavor_ids:
    percentages = np.where(total_weighted > 0, 100.0 * weighted_hists[fid] / safe_total, 0.0)
    plt.plot(energy_centers, percentages, marker=flavor_markers[fid], label=flavor_labels[fid])

plt.xscale("log")
plt.ylim(0, 100)
plt.xlabel("Energy (eV)")
plt.ylabel("Weight percentage (%)")
plt.title("Flavor percentage distribution vs energy (continuous spectrum)")
plt.grid(True, which="both", ls=":", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Log-spaced bins over passed-energy range
n_bins = 30
bins = np.logspace(np.log10(inelasticity_passed.min()), np.log10(inelasticity_passed.max()), n_bins + 1)

# Weighted histogram
hist_w, edges = np.histogram(inelasticity_passed, bins=bins, weights=w_passed)
centers = np.sqrt(edges[:-1] * edges[1:])  # geometric bin center

plt.figure(figsize=(8, 5))
plt.step(centers, hist_w, where="mid", linewidth=2, label="Passed weighted events")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Neutrino energy [eV]")
plt.ylabel("Weighted counts per bin")
plt.title("inelasticity_passed of passed weighted events")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
with h5py.File(h5_path, "r") as fin:
    energies = np.asarray(fin["energies"], dtype=float)
    weights = np.asarray(fin["weights"], dtype=float)
    triggered = np.asarray(fin["triggered"], dtype=bool)
    event_group_ids = np.asarray(fin["event_group_ids"])
    Emin = float(fin.attrs["Emin"])
    Emax = float(fin.attrs["Emax"])
    n_events_total = float(fin.attrs["n_events"])
    sim_volume = float(fin.attrs["volume"])


# log-energy binning over the continuous spectrum
n_bins = 12
energy_bins = np.logspace(np.log10(E_passed.min()), np.log10(E_passed.max()), n_bins + 1)
energy_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

gamma = 2.0  # matches spectrum="E-2"
energy_bins = np.logspace(np.log10(Emin), np.log10(Emax), n_bins + 1)
Elo = energy_bins[:-1]
Ehi = energy_bins[1:]
frac = (Elo**(1 - gamma) - Ehi**(1 - gamma)) / (Emin**(1 - gamma) - Emax**(1 - gamma))
n_events_per_bin = n_events_total * frac

# Sum event weights per energy bin
sum_w_per_bin, _ = np.histogram(E_passed, bins=energy_bins, weights=w_passed)

# Convert weighted fraction to Veff contribution per bin (then to water-equivalent km^3 sr)
veff_m3_per_bin = sim_volume * (sum_w_per_bin / n_events_per_bin)
veff_we_sr_per_bin = get_Veff_water_equivalent(veff_m3_per_bin) * 4 * np.pi

# Poisson-style relative uncertainty using effective triggered counts per bin
n_eff = np.where(sum_w_per_bin > 0, sum_w_per_bin, np.nan)
veff_err = np.where(np.isfinite(n_eff), veff_we_sr_per_bin / np.sqrt(n_eff), 0.0)

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.errorbar(
    energy_centers,
    veff_we_sr_per_bin / (units.km**3 * units.sr),
    yerr=veff_err / (units.km**3 * units.sr),
    fmt="o-",
    linewidth=2,
    markersize=6,
    label="Station 11 (merged continuous spectrum)",
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Neutrino energy [eV]")
ax.set_ylabel("effective volume [km$^3$ sr]")
ax.set_title("Binned Veff from merged continuous-spectrum simulation")
ax.grid(True, which="both", ls=":", alpha=0.5)
ax.legend()
fig.tight_layout()

for e, v in zip(energy_centers, veff_we_sr_per_bin):
    print(f"E = {e/units.eV:.2e} eV -> Veff = {v/(units.km**3 * units.sr):.3e} km^3 sr")

plt.show()


In [ ]:
xx_passed=xx[passed_mask]
yy_passed=yy[passed_mask]
zz_passed=zz[passed_mask]
zeniths_passed=zeniths[passed_mask]
azimuths_passed=azimuths[passed_mask]

#plot altitudes and azimuths on a scatter plot
plt.scatter(azimuths_passed*180/np.pi,zeniths_passed*180/np.pi,s=2)
plt.axhline(np.mean(zeniths)*180/np.pi , color='orange', linestyle='--', label="mean zenith value")
#add labels to plot
plt.title("Launch Angle of Neutrinos")
plt.legend()
plt.xlabel("Azimuth (degrees)")
plt.ylabel("Zenith (degrees)")

In [ ]:


det = NuRadioReco.detector.RNO_G.rnog_detector.Detector(select_stations = 11)
det.update(datetime.datetime(2024, 2, 1)) 
    # det.export("uncalibrated.json")
position = det.get_absolute_position(station_id = 11)
position_m = position / units.m
print(f"{position}")

#convert the cartesian coordinates to cylindrical
radial_distances=np.sqrt((xx_passed-position_m[0])**2+(yy_passed-position_m[1])**2)
log_radial_distances=np.log10(radial_distances)

#plot the cylindrical vertex distributions
#radial distance
fig,ax=plt.subplots()
ax.hist(log_radial_distances, log=True, bins=100)
ax.set_title('Radial Distance of Interaction Vertex')
ax.set_xlabel('log Distance (m)')
ax.set_ylabel('Frequency')
plt.show()

#azimuthal angle
fig,ax=plt.subplots()
ax.hist(azimuths_passed, bins=100)
ax.set_title('Azimuthal Angle of Interaction Vertex')
ax.set_xlabel('Azimuthal Angle (degrees)')
ax.set_ylabel('Frequency')
plt.show()
#depth
fig,ax=plt.subplots()
ax.hist(zz_passed, bins=100,orientation='horizontal')
ax.set_title('Depth of Interaction Vertex')
ax.set_xlabel('Frequency')
ax.set_ylabel('Depth (m)')
plt.show()

In [ ]:
#plot the interaction vertex in a 3d scatterplot
ax = plt.figure().add_subplot()
ax.scatter(np.log10(radial_distances),zz_passed)
ax.set_xlabel("r log(horizontal distance) (m)")
ax.set_ylabel("z distance (m)")
ax.set_title("Interaction Vertex Location")